In [1]:
import pandas as pd
from modAL.models import ActiveLearner
from modAL.models import CommitteeRegressor
from modAL.disagreement import vote_entropy_sampling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import torch
import kan
from kan import KAN, create_dataset_from_data

In [2]:
import numpy as np

In [3]:
torch.autograd.set_detect_anomaly(True)

In [4]:
df = pd.read_csv('ubend_gen_v4.csv')
for i in df.columns:
    if df[i].dtype is not np.float64:
        df[i] = df[i].astype(np.float64)
print(df.columns)
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
X_pool = df.drop("PT_LOSS", axis=1).values
y_pool = df["PT_LOSS"].values.reshape(-1, 1)
print(len(y_pool))
# print(X_pool,type(X_pool[100]))
# print(X_pool)
# tX, ty = torch.from_numpy(X_pool).float(), torch.from_numpy(y_pool).float()
# dataset = create_dataset_from_data(tX, ty)
# print(tX, ty)

Index(['B4TOD4', 'B5TOB4', 'R45TOB4', 'OMEGA5', 'GAMMAU', 'ALPHA4', 'RE',
       'PT_LOSS'],
      dtype='object')
1342


In [5]:
class KANWrapper:
    def __init__(self, **params):
        self.model = KAN(**params)

    def fit(self, X, y, opt="LBFGS", steps=80, lamb=0.001):
        dataset = create_dataset_from_data(X, y)
        self.model.fit(dataset, opt=opt, steps=steps, lamb=lamb)
        return self

    def predict(self, X):
        probs = self.model(X)
        return probs

In [8]:
def qbc(committee, X_sample):
    loss_fn = torch.nn.MSELoss()
    tx = torch.from_numpy(X_sample).float().requires_grad_(True)

    preds = []
    for model in committee.learner_list:
        pred = model.predict(tx)  # (1, 1) или (1,)
        preds.append(pred)

    preds = torch.stack(preds, dim=0)  # (N_models, 1, D) или (N_models, 1)
    f_avg = preds.mean(dim=0)         # (1, D)
    # MSE между всеми предсказаниями и средним
    loss = loss_fn(preds.squeeze(), f_avg.squeeze())
    return loss, tx

In [13]:
def NA_QBC(committee, X_sample):
    grads = []
    for i in range(X_sample.shape[0]):
        x_s = np.array([X_sample[i]])
        qbc_loss, tx = qbc(committee, x_s)
        if tx.grad is not None:
            tx.grad.zero_()
        qbc_loss.backward(retain_graph=True)
        grads.append(tx.grad.detach().clone())
    return grads

In [128]:
# def NA_query_strategy(comittee, X_sample):
#     grads = NA_QBC(comittee, X_sample) # (N, M) -> N
#     # x_gen = x + alpha * (grad - l_grad_bnd) # из статьи
#     steps = get_steps(x)
#     # x_gen = x + steps * sign(grad - l_grad_bnd)
#     for _ in range(grad_iter):
#         x_gen = x + steps * sign(grad - l_grad_bnd)
#         x = x_gen
#     return None, X_generated

def NA_query_strategy(comittee, X_sample):
    for i in range(2):
        grads = NA_QBC(comittee, X_sample) # (N, M) -> N
        l_grad_bnd = Lbnd(X_sample)
        # x_gen = x + alpha * (grad - l_grad_bnd) # из статьи
        result = [a - b for a, b in zip(grads, l_grad_bnd)]
        print(result, 'result grads - l_bnd')
        sign = [torch.where(tensor > 0, torch.tensor(1), 
                  torch.where(tensor < 0, torch.tensor(-1), torch.tensor(0))) for tensor in result]
        print(sign, 'sign grads - l_bnd')
        for ind in range(len(sign)):
            real_steps = []
            sign_list = sign[ind].squeeze().tolist()
            steps = get_steps(X_sample[ind])
            x_gen = None
            for index, values in enumerate(sign_list):
                real_steps.append(steps[index][values])
            print(real_steps,'real_steps')
            x_gen = X_sample[ind] + real_steps
            X_sample[ind] = x_gen.copy()
        # X_sample = X_sample[0]
        print(X_sample,"X_sample",i)
        # X_sample[ind] = x_gen.copy()
    # x_gen = x + steps * sign(grad - l_grad_bnd)
    # for ind in range(len(grads)):
    #     x_gen = x + steps * sign(grad[ind] - l_grad_bnd[ind])
    #     x = x_gen
    
    # return None, X_generated
    return None, X_sample

# steps = np.array([1, 0.2, ...])

In [122]:
def get_steps(x):
    steps = [{-1:None, 0: 0 , 1: None} for i in range(7)]
    
    if x[0] <= 0.008:
        steps[0][1] = 0.015
        steps[0][-1] = 0
    elif x[0] <= 0.025:
        steps[0][1] = 0.08
        steps[0][-1] = 0
    elif x[0] > 0.09:
        steps[0][1] = 0
        steps[0][-1] = -0.08

    if x[1] <= 1.15:
        steps[1][1] = 0.25
        steps[1][-1] = 0
    elif x[1] <= 1.4:
        steps[1][1] = 0.25
        steps[1][-1] = -0.25
    elif x[1] > 1.4:
        steps[1][1] = 0
        steps[1][-1] = -0.25

    if x[2] <= 1.15:
        steps[2][1] = 1.6
        steps[2][-1] = 0
    elif x[2] <= 2.7:
        steps[2][1] = 2.5
        steps[2][-1] = -1.6
    elif x[2] > 4.0:
        steps[2][1] = 0
        steps[2][-1] = -2.5

    if x[3] <= -1.15:
        steps[3][1] = 5.0
        steps[3][-1] = 0
    elif x[3] <= 2:
        steps[3][1] = 5.0
        steps[3][-1] = -5.0
    elif x[3] > 3:
        steps[3][1] = 0
        steps[3][-1] = -5.0

    if x[4] <= -0.25:
        steps[4][1] = 0.5
        steps[4][-1] = 0
    elif x[4] <= 0.35:
        steps[4][1] = 0.5
        steps[4][-1] = -0.5
    elif x[4] > 0.4:
        steps[4][1] = 0
        steps[4][-1] = -0.5

    if x[5] <= 15.0:
        steps[5][1] = 10.0
        steps[5][-1] = 0
    elif x[5] <= 25.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] <= 35.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] <= 45.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] <= 55.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] > 56.0:
        steps[5][1] = 0
        steps[5][-1] = -10.0

    if x[6] <= 150000.:
        steps[6][1] = 400000.
        steps[6][-1] = 0
    elif x[6] >= 400000.:
        steps[6][1] = 0.0
        steps[6][-1] = -400000.
    return steps
        


# [0.005 0.02  0.1  ] B4TOD4
# [1.25 1.   1.5 ] B5TOB4
# [0.9 5.  2.5] R45TOB4
# [-5.  5.  0.] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10. 30. 60. 20. 40. 50.] ALPHA4
# [100000. 500000.] RE

def Lbnd(X_sample, min_max_val =[[0.005,0.1],[1.,1.5],[0.9,5.],[-5.,5.],[-0.5,0.5],[10.,60.],[100000.,500000.]]):
    
    l_grad_bnd = []
    for i in range(X_sample.shape[0]):
        l_grad = []
        x_s = np.array([X_sample[i]])
        for ind in range(len(x_s[0])):
            if  x_s[0][ind] >= min_max_val[ind][1]:
                l_grad.append(-1.0)
            elif min_max_val[ind][0] < x_s[0][ind] < min_max_val[ind][1]:
                l_grad.append(0.0)
            elif  x_s[0][ind] <= min_max_val[ind][0]:
                l_grad.append(1.0)
        l_grad_bnd.append(torch.tensor([l_grad]))
    return l_grad_bnd
    

In [76]:
def get_new_y(X_sampels,X_pool,y_pool):
    index_list = []
    X_study = []
    y_study = []
    for target_row in X_sampels:
        # print(X_sampels)
        print(target_row)
        index = np.where((X_pool == target_row).all(axis=1))[0]
        # index = np.where(np.all(X_pool == target_row, axis=1))[0]
        if index.size > 0:
            print(f"Найдена строка {target_row} на индексе {index[0]}")
            if index[0] not in index_list:
                X_study.append(target_row)
                y_study.append(y_pool[index[0]])
                index_list.append(index[0])
        else:
            print(f"Строка {target_row} не найдена в X_pool")
            # while index.size < 0:
            #     X_sam1 = NA_query_strategy(comittee, [target_row])
            #     index = np.where((X_pool == X_sam1[0]).all(axis=1))[0]
            # X_study.append(target_row)
            # y_study.append(y_pool[index[0]])
            # index_list.append(index[0])
    return X_study, y_study, index_list

In [132]:
train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
X_train = X_pool[train_idx]
y_train = y_pool[train_idx]
print(X_train,y_train,type(X_train),type(y_train))
_, X_generated = committee.query(X_train)
print(X_generated)
# x_st, y_st, ind = get_new_y(X_generated,X_pool,y_pool)
# X_pool = np.delete(X_pool, ind, axis=0)
# y_pool = np.delete(y_pool, ind)
# committee.teach(
#     x_st.reshape(1, -1),
#     y_st.reshape(1, -1),
# )

[[ 2.00e-02  1.50e+00  9.00e-01  0.00e+00 -5.00e-01  1.00e+01  1.00e+05]
 [ 5.00e-03  1.50e+00  2.50e+00  0.00e+00  0.00e+00  6.00e+01  1.00e+05]
 [ 5.00e-03  1.50e+00  5.00e+00  0.00e+00  5.00e-01  5.00e+01  1.00e+05]
 [ 5.00e-03  1.25e+00  2.50e+00  0.00e+00  0.00e+00  1.00e+01  1.00e+05]
 [ 2.00e-02  1.00e+00  9.00e-01  5.00e+00  0.00e+00  3.00e+01  1.00e+05]
 [ 1.00e-01  1.00e+00  9.00e-01 -5.00e+00  5.00e-01  1.00e+01  5.00e+05]
 [ 1.00e-01  1.25e+00  5.00e+00  0.00e+00  5.00e-01  4.00e+01  1.00e+05]] [[0.1528   ]
 [0.06525  ]
 [0.1688   ]
 [0.27     ]
 [0.0740551]
 [0.169028 ]
 [0.3987   ]] <class 'numpy.ndarray'> <class 'numpy.ndarray'>


C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:813: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  self.subnode_actscale.append(torch.std(x, dim=0).detach())
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:823: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  input_range = torch.std(preacts, dim=0) + 0.1
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:824: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\act

[tensor([[-0.8192, -2.1656, -3.9354, -1.4109,  1.0625,  9.3801,  2.7372]]), tensor([[-1.8044, -2.1699, -3.9588, -1.4128,  4.6012, 11.3902,  2.7423]]), tensor([[-1.8042, -2.1690, -2.6981, -1.4124,  7.2000, 10.3871,  2.7412]]), tensor([[-1.8033, -3.1437, -3.9533, -1.4109,  4.5948,  9.3800,  2.7371]]), tensor([[-0.8196, -3.4363, -3.9370, -0.0453,  4.5974, 10.3816,  2.7392]]), tensor([[ -3.8645, -13.1721, -15.6738,  -0.8649,  31.9602,  50.8902,  19.6820]]), tensor([[ 0.0261, -3.1463, -2.6971, -1.4120,  7.1983, 10.3842,  2.7402]])] result grads - l_bnd
[tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[ 1, -1, -1, -1,  1,  1,  1]])] sign grads - l_bnd
[0, -0.25, 0, -5.0, 0.5, 10.0, 400000.0] real_steps
[0, -0.25, -1.6, -5.0, 0.5, 0, 400000.0] real_steps
[0, -0.25, -2.5, -5.0, 0, 10.0, 400000.0] 

In [134]:
print(X_generated[0])

[[ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05]
 [ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05]
 [ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05]
 [ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05]
 [ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  5.e+01  5.e+05]
 [ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05]
 [ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05]]


In [140]:
# print(X_generated[0],type(X_generated))
x_st, y_st, ind = get_new_y(X_generated[0],X_pool,y_pool)
print(x_st,y_st)
# X_pool = np.delete(X_pool, ind, axis=0)
# y_pool = np.delete(y_pool, ind)
x_st = x_st[0][np.newaxis, :]
y_st = y_st[0][np.newaxis, :]
tX, ty = torch.from_numpy(x_st).float(), torch.from_numpy(y_st).float()
for ind in range(len(x_st)):
    # print(x_st[ind].reshape(1, -1),y_st[ind].reshape(1, -1))
    print(tX,ty)
    committee.teach(
        # x_st[ind].reshape(1, -1),
        # y_st[ind].reshape(1, -1),
        # x_st[ind],
        # y_st[ind],
        tX,
        ty, bootstrap=True
    )

[ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05]
Найдена строка [ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05] на индексе 236
[ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05]
Строка [ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05] не найдена в X_pool
[ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05]
Строка [ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05] не найдена в X_pool
[ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05]
Строка [ 5.e-03  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05] не найдена в X_pool
[ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  5.e+01  5.e+05]
Строка [ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  5.e+01  5.e+05] не найдена в X_pool
[ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05]
Строка [ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  3.e+01  5.e+05] не найдена в X_pool
[ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05]
Строка [ 2.e-02  1.e+00  9.e-01 -5.e+00  5.e-01  6.e+01  5.e+05] не найдена в X_p

description:   0%|                                                           | 0/80 [00:00<?, ?it/s]C:\Users\ivan\anaconda3\Lib\site-packages\torch\autograd\graph.py:823: UserWarning: Error detected in MulBackward0. Traceback of forward call that caused the error:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ivan\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\ivan\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\ivan\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\ivan\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\ivan\anaconda3\Lib\asyncio\windows_events.py", line 322, in run_forever
    super().run_forever()
  File

RuntimeError: Function 'MulBackward0' returned nan values in its 1th output.

In [130]:
n_members = 2  # количесво моделей
learner_list = list()
grid1 = [3, 7]
k1 = [5, 3]
n_queries = 10  # Количество итераций активного обучения
# for i in range(n_queries):
for member_idx in range(n_members):
    # initial training data
    n_initial = 7
    train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
    # X_train = X_pool[train_idx]
    # y_train = y_pool[train_idx]
    # tX, ty = torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()
    # # creating a reduced copy of the data with the known instances removed
    # X_pool = np.delete(X_pool, train_idx, axis=0)
    # y_pool = np.delete(y_pool, train_idx)

    # initializing learner
    learner = ActiveLearner(
        estimator=KANWrapper(
            width=[7, 7, 7, 1], grid=grid1[member_idx], k=k1[member_idx], seed=42
        ),  # вот сюда засовываем наш KAN
    )
    learner_list.append(learner)

# assembling the committee

committee = CommitteeRegressor(learner_list=learner_list, query_strategy=NA_query_strategy)

checkpoint directory created: ./model
saving model version 0.0
checkpoint directory created: ./model
saving model version 0.0


In [65]:
grad_ = NA_QBC(committee, X_pool[:3])

C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:813: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  self.subnode_actscale.append(torch.std(x, dim=0).detach())
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:823: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  input_range = torch.std(preacts, dim=0) + 0.1
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:824: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\act

In [66]:
print(grad_)


[tensor([[ -4.8648, -15.7164, -18.4676,  -7.0533,  30.9617,  51.8714,  18.6829]]), tensor([[-0.8194, -3.1445, -3.6950, -1.4112,  4.5960, 10.3784,  3.7381]]), tensor([[ -4.8651, -12.1734, -14.6754,  -5.2232,  22.9722,  51.8745,  18.6840]])]


In [32]:
print(y_pool[100],type(y_pool[100]))
print(X_pool[100],type(X_pool[100]))
model1 = KANWrapper(width=[7, 7, 7, 1], grid=3, k=5,seed=42)
model1.fit(tX,ty)


0.115285 <class 'numpy.float64'>
[ 5.00e-03  1.25e+00  9.00e-01 -5.00e+00  0.00e+00  1.00e+01  1.00e+05] <class 'numpy.ndarray'>
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 7.17e-02 | test_loss: 1.46e-01 | reg: 1.36e+01 | : 100%|█| 80/80 [08:09<00:00,  6.12s/

saving model version 0.1


In [35]:
tx1 = torch.Tensor([X_pool[100].tolist()]).float().requires_grad_(True)
ty1 = torch.Tensor([[y_pool[100].item()]]).float()
pred = model1.predict(tx1)

C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:813: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  self.subnode_actscale.append(torch.std(x, dim=0).detach())
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:823: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  input_range = torch.std(preacts, dim=0) + 0.1
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:824: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\act

In [39]:
loss_fn = torch.nn.MSELoss()
loss = loss_fn(pred, ty1)
loss.backward()
# Получаем производные по входным данным
gradients = tx1.grad
print(gradients)

C:\Users\ivan\anaconda3\Lib\site-packages\torch\autograd\graph.py:823: UserWarning: Error detected in MulBackward0. Traceback of forward call that caused the error:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ivan\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\ivan\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\ivan\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\ivan\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\ivan\anaconda3\Lib\asyncio\windows_events.py", line 322, in run_forever
    super().run_forever()
  File "C:\Users\ivan\anaconda3\Lib\asyncio\base_events.py", line 641, in run_forever
    self._run_once()

RuntimeError: Function 'MulBackward0' returned nan values in its 1th output.

In [ ]:
# n_queries = 10  # Количество итераций активного обучения
# n_committee = 5  # Количество моделей в комитете
# # Основной цикл активного обучения
# for i in range(n_queries):
#     # Создание комитета моделей
#     committee = [RandomForestRegressor() for _ in range(n_committee)]
    
#     # Обучение моделей на текущем наборе данных
#     for model in committee:
#         model.fit(X_train, y_train)
    
#     # Получение предсказаний от всех моделей
#     predictions = np.array([model.predict(X_pool) for model in committee])

# # Вычисление неопределенности (разброс предсказаний)
#     uncertainty = np.std(predictions, axis=0)
    
#     # Выбор экземпляра с наибольшей неопределенностью
#     query_index = np.argmax(uncertainty)
    
#     # Добавление выбранного экземпляра в обучающую выборку
#     X_train = np.vstack((X_train, X_pool[query_index].reshape(1, -1)))
#     y_train = np.append(y_train, y_pool[query_index])
    
#     # Удаление выбранного экземпляра из пула
#     X_pool = np.delete(X_pool, query_index, axis=0)
#     y_pool = np.delete(y_pool, query_index)

In [ ]:
# def qbc(committee, X_pool, y_pool, n_initial):
#     qsum = [0 for i in range(len(n_initial)]
#     f_average = [0 for i in range(len(n_initial)]
#     train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
#     Sbp = 2
#     X_add = X_pool[train_idx]
#     y_add = y_pool[train_idx]
#     for i in range(n_initial):
#         sample = X_add[i]
#         tx = torch.from_numpy(sample).float() 
#         for model_ind in range(len(committee)):
#             f_average[n_initial] += committee[model_ind].predict(tx)
#         f_average[n_initial] = f_average[n_initial]/len(committee)
#     for i in range(n_initial):
#         sample = X_add[i]
#         for model in committee:
#             qsum[i] += (model.predict(tx) - f_average[i])**2
#         qsum[i] /= len(committee)
#     return qsum